# Transformer 500M — self-resuming training

**Target: 539,502,592 parameters, ~10–12 days on free/cheap GPUs.**

1. **Runtime → Change runtime type → GPU** (Colab) or attach a GPU accelerator (Kaggle). A T4 (16 GB) works; an A100/L4 finishes much faster.
2. Run the cells top to bottom. Cell 6 loops training in 2-hour chunks with **auto-resume** — if the session disconnects, just re-run Cell 6. Checkpoints/status are copied to **Google Drive** (Colab) each chunk.
3. A supervisor agent reads `transformer-500m/status.json` from Drive and reports progress.

Data: TinyStories (2.3 GB) + WikiText-103 (0.5 GB) → ~700M tokens, 32k BPE. With ~2.6B total tokens (multiple epochs / larger corpus flag below), expect a solid base model for this size class.

In [ ]:
#@title 1 · Setup (GPU check, repo, deps)
import torch, os, subprocess, sys, shutil, json, time
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('GPU:', gpu or 'NONE — enable it: Runtime > Change runtime type > GPU')
assert gpu, 'A GPU is required for the 500M run'
print('bf16 supported:', torch.cuda.is_bf16_supported())

REPO = '/content/transformer-llm'
IN_COLAB = 'google.colab' in sys.modules
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/samratbarman1013-commits/transformer-llm.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tokenizers', 'numpy', 'onnx', 'onnxruntime'], check=True)
os.chdir(REPO)
print('repo ready:', os.getcwd())

In [ ]:
#@title 2 · Mount Drive (checkpoint home) — Colab only
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/transformer-500m'
else:
    DRIVE = '/kaggle/working/transformer-500m'  # Kaggle: survives via Output dataset
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
STATUS = f'{DRIVE}/status.json'
print('checkpoint home:', DRIVE)

In [ ]:
#@title 3 · Download the corpus (~2.8 GB, ~5 min)
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)
ts = f'{DATA}/tinystories.txt'
if not os.path.exists(ts):
    !wget -q -O {ts} "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt"
print('TinyStories:', os.path.getsize(ts)/1e9, 'GB')
wt = f'{DATA}/wikitext.txt'
if not os.path.exists(wt):
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
        from datasets import load_dataset
        ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train')
        with open(wt, 'w') as f:
            for row in ds:
                f.write(row['text'] + '\n')
    except Exception as e:
        print('wikitext skipped:', e)
print('corpus ready')

In [ ]:
#@title 4 · Tokenizer (32k BPE) — trained once, reused across sessions
from transformer.tokenizer_bpe import train as train_tok, encode_file
TOK = f'{DRIVE}/tokenizer.json'
if not os.path.exists(TOK):
    train_tok(ts, TOK, vocab_size=32768)   # TinyStories dominates; representative
else:
    print('reusing tokenizer from Drive')

In [ ]:
#@title 5 · Pre-tokenize to train.bin / val.bin (~15 min, once)
BINS = f'{DATA}/bins'
if not os.path.exists(f'{BINS}/train.bin'):
    # combine TinyStories + WikiText into one stream, then encode
    combined = f'{DATA}/corpus.txt'
    if not os.path.exists(combined):
        with open(combined, 'w') as out:
            for src in (ts, wt):
                if os.path.exists(src):
                    with open(src) as f: shutil.copyfileobj(f, out)
    encode_file(combined, TOK, BINS, val_frac=0.001)
print(json.load(open(f'{BINS}/meta.json')))

In [ ]:
#@title 6 · TRAIN (auto-resume loop — re-run this cell after any disconnect)
TOTAL_STEPS = 40000        # ~2.6B tokens at 65k tokens/step
CHUNK_SECONDS = 2 * 3600   # checkpoint to Drive every 2 hours
CKPT = '/content/ckpt'     # fast local disk; mirrored to Drive each chunk

def latest_step():
    try: return json.load(open(STATUS)).get('step', 0)
    except Exception: return 0

n_gpu = torch.cuda.device_count()
while latest_step() < TOTAL_STEPS:
    base = ([sys.executable] if n_gpu == 1 else
            ['torchrun', f'--nproc_per_node={n_gpu}'])
    cmd = base + ['-m', 'transformer.train_v2',
        '--config', 'target-500m', '--data', BINS, '--out', CKPT,
        '--steps', str(TOTAL_STEPS), '--batch', '4', '--grad-accum', '16',
        '--ctx', '1024', '--lr', '2e-4', '--warmup', '400',
        '--eval-every', '500', '--grad-checkpoint', '--dropout', '0.0',
        '--max-seconds', str(CHUNK_SECONDS), '--status-out', STATUS]
    print('>> launching:', ' '.join(cmd))
    p = subprocess.run(cmd)
    for f in ('ckpt_last.pt', 'model_best.pt', 'train_log.jsonl'):
        src = f'{CKPT}/{f}'
        if os.path.exists(src): shutil.copy2(src, f'{DRIVE}/checkpoints/{f}')
    s = latest_step()
    print(f'chunk done (rc={p.returncode}) — step {s}/{TOTAL_STEPS}')
    if p.returncode != 0:
        time.sleep(30)  # transient GPU reset; loop retries
print('TRAINING COMPLETE — best model in', f'{DRIVE}/checkpoints/model_best.pt')

In [ ]:
#@title 7 · Chat with the trained 500M model
import torch
from transformer.config import CONFIGS
from transformer.model import GPT
from transformer.tokenizer_bpe import BpeTokenizer

ck = torch.load(f'{DRIVE}/checkpoints/model_best.pt', map_location='cpu', weights_only=False)
cfg = CONFIGS[ck['config']]
cfg.gradient_checkpointing = False
model = GPT(cfg, ck['vocab_size']); model.load_state_dict(ck['model']); model.eval()
tok = BpeTokenizer(f'{DRIVE}/tokenizer.json')
prompt = 'Once upon a time there was a little girl named Lily'  #@param {type:"string"}
ids = torch.tensor([tok.encode(prompt)], dtype=torch.long)
out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_k=50)[0].tolist()
print(tok.decode(out[len(ids[0]):]))

## When training completes

- Export for the browser/phone: `python -m transformer.export_onnx --checkpoint <model_best.pt> --out web/model/model.onnx` (then re-deploy the site — see `TRAINING_500M.md`)
- Publish the weights: upload `model_best.pt` to the GitHub release (the supervisor can help)
- Next milestones: SFT on chat data (DPO later) — see `TRAINING_500M.md`